# Questão 3 — Compressão de áudio com DFT e DCT

Este notebook apresenta a implementação em Python utilizada para resolver a Questão 3 da Aula Prática 4 de Processamento de Sinais I.

## Objetivo

Comparar o desempenho da **DFT** e da **DCT** na compressão do sinal de áudio `handel.wav`, mantendo diferentes frações da energia total do sinal.

Os fatores de preservação de energia analisados são

$$
r \in \{99{,}5\%,\ 99{,}0\%,\ 90{,}0\%,\ 75{,}0\%,\ 50{,}0\%\}.
$$

A análise considera:

- quantidade de coeficientes mantidos;
- percentual de coeficientes preservados;
- fator de compressão;
- erro quadrático médio da reconstrução.

## Bibliotecas utilizadas

Serão utilizadas as bibliotecas:

- `NumPy` para cálculos numéricos;
- `Matplotlib` para geração de gráficos;
- `SciPy` para leitura do arquivo de áudio e aplicação da DCT.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.fft import dct, idct

## Leitura do arquivo de áudio

Nesta etapa, o arquivo `handel.wav` é carregado para análise.

Se o sinal possuir mais de um canal, ele será convertido para um vetor unidimensional para simplificar o processamento.

In [ ]:
fs, x = wavfile.read("../dados/handel.wav")
x = x.astype(float)

if x.ndim > 1:
    x = x[:, 0]

N = len(x)

print("Frequência de amostragem:", fs, "Hz")
print("Número de amostras:", N)
print("Tipo do sinal:", x.dtype)

## Definição da energia e do erro

A energia dos coeficientes transformados é calculada por

$$
E = \sum_k |C[k]|^2,
$$

em que \(C[k]\) representa os coeficientes da DFT ou da DCT.

A qualidade da reconstrução será avaliada pelo erro quadrático médio (EQM), definido por

$$
EQM = \frac{1}{N}\sum_{n=0}^{N-1}\left(x[n] - \hat{x}[n]\right)^2,
$$

onde \(x[n]\) é o sinal original e \(\hat{x}[n]\) é o sinal reconstruído.

## Função de compressão com DCT

A função abaixo calcula a DCT do sinal, ordena os coeficientes pela energia, mantém apenas os coeficientes necessários para atingir a fração de energia desejada e reconstrói o sinal por meio da DCT inversa.

In [ ]:
def compress_dct(x, r):
    C = dct(x, norm='ortho')
    energy = np.abs(C) ** 2

    idx = np.argsort(energy)[::-1]
    sorted_energy = energy[idx]
    cum_energy = np.cumsum(sorted_energy)
    total_energy = cum_energy[-1]

    k = np.searchsorted(cum_energy, r * total_energy) + 1

    Cc = np.zeros_like(C)
    Cc[idx[:k]] = C[idx[:k]]

    x_rec = idct(Cc, norm='ortho')
    mse = np.mean((x - x_rec) ** 2)

    return x_rec, k, mse

## Função de compressão com DFT

Para a DFT, o sinal é transformado para o domínio da frequência por meio da FFT. Em seguida, são mantidos os coeficientes mais energéticos até atingir a fração de energia desejada.

Como o sinal é real, a reconstrução utiliza a transformada inversa e considera a parte real do resultado.

In [ ]:
def compress_dft(x, r):
    X = np.fft.fft(x)
    energy = np.abs(X) ** 2

    idx = np.argsort(energy)[::-1]
    sorted_energy = energy[idx]
    cum_energy = np.cumsum(sorted_energy)
    total_energy = cum_energy[-1]

    k = np.searchsorted(cum_energy, r * total_energy) + 1

    Xc = np.zeros_like(X)
    Xc[idx[:k]] = X[idx[:k]]

    x_rec = np.fft.ifft(Xc).real
    mse = np.mean((x - x_rec) ** 2)

    return x_rec, k, mse

## Taxas de energia preservada

As taxas de preservação de energia utilizadas no experimento são:

$$
r = \{0{,}995,\ 0{,}99,\ 0{,}90,\ 0{,}75,\ 0{,}50\}.
$$

A célula seguinte executa a compressão do áudio para cada caso e armazena os resultados para posterior comparação.

In [ ]:
r_values = [0.995, 0.99, 0.90, 0.75, 0.50]

results = []

for r in r_values:
    x_dft, k_dft, mse_dft = compress_dft(x, r)
    x_dct, k_dct, mse_dct = compress_dct(x, r)

    pct_dft = 100 * k_dft / N
    pct_dct = 100 * k_dct / N

    comp_dft = N / k_dft
    comp_dct = N / k_dct

    results.append({
        "r": r,
        "k_dft": k_dft,
        "pct_dft": pct_dft,
        "comp_dft": comp_dft,
        "mse_dft": mse_dft,
        "k_dct": k_dct,
        "pct_dct": pct_dct,
        "comp_dct": comp_dct,
        "mse_dct": mse_dct
    })

## Exibição dos resultados numéricos

A tabela abaixo mostra, para cada valor de \(r\):

- número de coeficientes mantidos;
- percentual de coeficientes preservados;
- fator de compressão aproximado;
- erro quadrático médio.

Essa comparação permite avaliar qual transformada concentra melhor a energia do sinal.

In [ ]:
print(f"{'r':>8} | {'DFT k':>8} | {'DFT %':>8} | {'DFT N/k':>10} | {'DFT EQM':>12} | {'DCT k':>8} | {'DCT %':>8} | {'DCT N/k':>10} | {'DCT EQM':>12}")
print("-" * 110)

for res in results:
    print(f"{100*res['r']:7.1f}% | "
          f"{res['k_dft']:8d} | "
          f"{res['pct_dft']:7.2f}% | "
          f"{res['comp_dft']:10.2f} | "
          f"{res['mse_dft']:12.2f} | "
          f"{res['k_dct']:8d} | "
          f"{res['pct_dct']:7.2f}% | "
          f"{res['comp_dct']:10.2f} | "
          f"{res['mse_dct']:12.2f}")

## Gráfico 1 — Número de coeficientes mantidos

Este gráfico mostra quantos coeficientes precisam ser mantidos na DFT e na DCT para preservar cada valor de energia \(r\).

Quanto menor o número de coeficientes necessários, maior a capacidade de concentração de energia da transformada.

In [ ]:
r_plot = [100 * res["r"] for res in results]
k_dft_plot = [res["k_dft"] for res in results]
k_dct_plot = [res["k_dct"] for res in results]

plt.figure(figsize=(8, 5))
plt.plot(r_plot, k_dft_plot, marker='o', label='DFT')
plt.plot(r_plot, k_dct_plot, marker='o', label='DCT')
plt.xlabel('Energia preservada r (%)')
plt.ylabel('Número de coeficientes mantidos')
plt.title('Coeficientes necessários para compressão')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Gráfico 2 — Percentual de coeficientes mantidos

Este gráfico mostra o percentual de coeficientes mantidos em relação ao total de amostras do sinal.

In [ ]:
pct_dft_plot = [res["pct_dft"] for res in results]
pct_dct_plot = [res["pct_dct"] for res in results]

plt.figure(figsize=(8, 5))
plt.plot(r_plot, pct_dft_plot, marker='o', label='DFT')
plt.plot(r_plot, pct_dct_plot, marker='o', label='DCT')
plt.xlabel('Energia preservada r (%)')
plt.ylabel('Coeficientes mantidos (%)')
plt.title('Percentual de coeficientes mantidos')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Gráfico 3 — Erro quadrático médio

O gráfico abaixo mostra como o erro quadrático médio varia com a taxa de energia preservada. Espera-se que o erro aumente à medida que a compressão se torna mais agressiva.

In [ ]:
mse_dft_plot = [res["mse_dft"] for res in results]
mse_dct_plot = [res["mse_dct"] for res in results]

plt.figure(figsize=(8, 5))
plt.plot(r_plot, mse_dft_plot, marker='o', label='DFT')
plt.plot(r_plot, mse_dct_plot, marker='o', label='DCT')
plt.xlabel('Energia preservada r (%)')
plt.ylabel('EQM')
plt.title('Erro quadrático médio da reconstrução')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Gráfico 4 — Comparação de um trecho do sinal reconstruído

Para visualizar qualitativamente a reconstrução, será comparado um pequeno trecho do sinal original com os sinais reconstruídos por DFT e DCT para o caso

$$
r = 75\%.
$$

In [ ]:
x_dft_75, _, _ = compress_dft(x, 0.75)
x_dct_75, _, _ = compress_dct(x, 0.75)

samples = np.arange(0, min(2000, N))
t = samples / fs

plt.figure(figsize=(9, 5))
plt.plot(t, x[samples], label='Original')
plt.plot(t, x_dft_75[samples], label='Reconstruído DFT (75%)')
plt.plot(t, x_dct_75[samples], label='Reconstruído DCT (75%)')
plt.xlabel('Tempo (s)')
plt.ylabel('Amplitude')
plt.title('Trecho do sinal reconstruído para r = 75%')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Conclusão

Os resultados mostram que, para todas as taxas de preservação de energia analisadas, a **DCT** necessita de menos coeficientes do que a **DFT** para atingir a mesma fração de energia do sinal.

Isso indica que a DCT apresenta maior capacidade de concentração de energia, sendo mais eficiente para compressão de sinais de áudio.

Além disso:

- o erro quadrático médio aumenta à medida que a compressão se torna mais agressiva;
- para taxas altas de energia preservada, a reconstrução tende a ficar muito próxima do sinal original;
- para taxas menores, a degradação aumenta progressivamente.

Assim, conclui-se que, para o áudio analisado, a **DCT** apresenta desempenho superior ao da **DFT** na compressão por retenção dos coeficientes mais energéticos.